# 07 · Drought, water balance, and accumulation

**Atlas source reading copy · browser-network**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/07-drought.ipynb)



Browser-only piplite prepares numpy, pandas, matplotlib and Plotly 6.5.0; native kernels use installed scientific packages. Local calculations use bundled helpers and inputs; Plotly rendering can require its versioned CDN and WebGL. Seeded monthly examples are synthetic. The C3S warming-level table gives model windows, Natural Earth provides real generalized coastlines, and the CDS catalogue is metadata. No authenticated CDS request, archive download or native regridding is run.

**Provenance:** `content/07-drought.ipynb` at `b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a`; original SHA-256 `27dc10f5aeb3fb3ab5cdbc29c954643cfd51e54765786046ab9cab65654f99d9`. Original course material MIT; C3S auxiliary table Apache-2.0, Natural Earth public domain and CDS catalogue metadata CC BY 4.0. Preserve source credits, licenses and artwork provenance.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Original climate-globe illustration introducing Drought, water balance, and accumulation; illustrative, not a data map](assets/atlas-splash.png)

# 07 · Drought, water balance, and accumulation

**Guiding question:** Why do drought indicators depend on both timescale and reference climate?

**Data:** Synthetic teaching examples  
**Time:** 20–35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.  
**Author:** JupyterLite-C3S-Atlas contributors (AI assisted) · **Created/updated:** 2026-11-15  
**Inputs:** bundled reference files and/or seeded examples specified below. **Outputs:** in-memory tables and figures; source data are not modified.  
**Runtime:** Python/Pyodide; NumPy, pandas, Matplotlib, Plotly; versions and tests are recorded in the validation report.  
**Change log:** initial educational chapter. **Contact:** [repository issues](https://github.com/jltobias/JupyterLite-C3S-Atlas/issues).

[Open in JupyterLite Lab](https://jltobias.github.io/JupyterLite-C3S-Atlas/lite/lab/index.html?path=07-drought.ipynb) · [Official Atlas](https://atlas.climate.copernicus.eu/)


## Read the evidence

Meteorological, agricultural, and hydrological drought refer to different
       parts of the system. Six-month precipitation and precipitation minus potential
       evapotranspiration describe accumulated supply and atmospheric demand. Soil moisture
       and runoff offer different evidence.

       The calculation here is a six-month water-balance sum, **not SPI-6 or SPEI-6**.
       Those indices need distribution fitting and a specified calibration period.
       Standardising with a simple mean and standard deviation is not a faithful substitute.
       Consult the upstream implementation for the current PET method and calibration.

## Start the browser lab

Run this setup cell once. First load needs internet for the Python runtime and packages. The remaining lesson uses bundled data. If a 3D view is unavailable, use the 2D/table alternative. Download your edited notebook before clearing browser storage.

In [1]:
# Install only in the browser kernel; desktop dependencies come from requirements.txt.
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "pandas", "matplotlib", "plotly==6.5.0"])

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from atlas_helpers import (
    DATA, SYNTHETIC, COLORS, show, monthly_demo, anomalies, regular_grid,
    area_mean, coastlines, map_figure, globe_figure, warming_levels, load_monthly_csv,
)
print("Ready. Browser edits live in this browser; download work you want to keep.")


Ready. Browser edits live in this browser; download work you want to keep.


## Track accumulated deficit

In [2]:
df = monthly_demo()
df["balance_mm"] = (df.r - df.pet) * df.days
df["balance_6month_mm"] = df.balance_mm.rolling(6, min_periods=6).sum()
assert df.balance_6month_mm.iloc[:5].isna().all()
fig = go.Figure(go.Scatter(x=df.date, y=df.balance_6month_mm, fill="tozeroy", name="Six-month P − PET"))
fig.add_hline(y=0, line_dash="dot")
fig.update_layout(title="Accumulated water balance, not SPEI · SYNTHETIC", xaxis_title="Month", yaxis_title="Six-month water balance (mm)")
show(fig)

Figure({
    'data': [{'fill': 'tozeroy',
              'name': 'Six-month P − PET',
              'type': 'scatter',
              'x': array(['1961-01-01T00:00:00.000000000', '1961-02-01T00:00:00.000000000',
                          '1961-03-01T00:00:00.000000000', ..., '2020-10-01T00:00:00.000000000',
                          '2020-11-01T00:00:00.000000000', '2020-12-01T00:00:00.000000000'],
                         shape=(720,), dtype='datetime64[ns]'),
              'y': {'bdata': ('AAAAAAAA+H8AAAAAAAD4fwAAAAAAAP' ... 'uthcR7eMBtQ+Tj1rRzwJcWzk7TCmTA'),
                    'dtype': 'f8'}}],
    'layout': {'font': {'size': 14},
               'height': 470,
               'margin': {'b': 65, 'l': 65, 'r': 45, 't': 90},
               'shapes': [{'line': {'dash': 'dot'},
                           'type': 'line',
                           'x0': 0,
                           'x1': 1,
                           'xref': 'x domain',
                           'y0': 0,
                           'y1': 0,
                           'yref': 'y'}],
               'template': '...',
               'title': {'text': 'Accumulated water balance, not SPEI · SYNTHETIC'},
               'xaxis': {'title': {'text': 'Month'}},
               'yaxis': {'title': {'text': 'Six-month water balance (mm)'}}}
})

## Count a dry spell without erasing missing days

In [3]:
# A small hand-checkable sequence. Missing breaks a confirmed spell and must be reported.
daily_rain = np.array([0.0, 0.2, 1.0, 0.0, 0.0, 0.0, 4.0])
run = longest = 0
for amount in daily_rain:
    run = run + 1 if np.isfinite(amount) and amount < 1 else 0
    longest = max(longest, run)
print("Longest confirmed dry spell:", longest, "days")
assert longest == 3
print("On incomplete real daily data this is a lower bound, not a complete annual CDD.")

Longest confirmed dry spell: 3 days
On incomplete real daily data this is a lower bound, not a complete annual CDD.


## Try it and explain

Change the accumulation from 6 to 12 months. Why does the response become smoother?

**Interpretation check:** Longer windows integrate more months, spreading short disturbances over time. Overlapping windows are strongly dependent; they are not independent samples.

## Transfer to the official Atlas

Compare SPI-6, SPEI-6, soil moisture, and runoff where available. Record both the index calibration period and the display baseline.

Record the selections, source version, units, periods, and uncertainty legend. These independent lessons do not duplicate every production workflow in the official tools.

## Sources and credit

- [C3S Atlas dataset and DOI](https://doi.org/10.24381/cds.h35hb680)
- [CSIC / Predictia Atlas user guide](https://confluence.ecmwf.int/spaces/CKB/pages/394237545/Copernicus+Interactive+Climate+Atlas+User+Guide)
- [Official C3S Atlas User Tools and contributors](https://ecmwf-projects.github.io/c3s-atlas/)
- [Upstream drought methods](https://github.com/ecmwf-projects/c3s-atlas/blob/main/book/notebooks/spei6.ipynb)

Original teaching text and code; original AI-generated splash illustration. Real reference files retain their source licences. Synthetic examples are not Copernicus measurements. See [credits](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/credits.md) and [data guide](https://github.com/jltobias/JupyterLite-C3S-Atlas/blob/b41c3e1098a4fb94cef48b3044fe19c6ce7c1a1a/content/data-guide.md).

**Continue:** [Wind, radiation, and energy context](08-wind-and-radiation.ipynb)